# 05 Final Load Prep

**Purpose:** Compute final KPIs, engineer Tableau-ready features, and export the exact dataset that will be loaded into Tableau Public for the dashboard.

**Problem Statement:** Which diabetic patients are at high risk of being readmitted within 30 days, and what clinical or demographic factors are most strongly associated with early readmission?

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
DATA_PATH = PROJECT_ROOT / 'data/processed/diabetic_data_clean.csv'
TABLEAU_PATH = PROJECT_ROOT / 'data/processed/tableau_ready.csv'

df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")

Loaded: 69,990 rows, 48 columns


## 1. Handle Remaining Nulls
Fill any remaining nulls in the admission/discharge ID columns so Tableau filters don't break.

In [2]:
null_cols = df.isnull().sum()
null_cols = null_cols[null_cols > 0]
if len(null_cols) > 0:
    print("Columns with nulls before fix:")
    print(null_cols)
    for col in null_cols.index:
        df[col] = df[col].fillna('Unknown')
    print("\n✅ All nulls filled with 'Unknown'")
else:
    print("✅ No nulls found")

print(f"\nNull check after fix: {df.isnull().sum().sum()} total nulls")

Columns with nulls before fix:
admission_type_id           4516
discharge_disposition_id    2474
admission_source_id         4821
dtype: int64

✅ All nulls filled with 'Unknown'

Null check after fix: 0 total nulls


## 2. Compute KPI Columns
Engineered features that will power the Tableau dashboard filters and calculations.

In [3]:
# KPI 1: Readmission Risk Label (for Tableau color coding)
df['readmission_status'] = df['readmitted_30day'].map({0: 'Not Readmitted', 1: 'Readmitted (<30 days)'})

# KPI 2: Age Risk Group
def age_risk(age):
    if age < 40: return 'Low Risk (<40)'
    elif age < 60: return 'Medium Risk (40-59)'
    elif age < 75: return 'High Risk (60-74)'
    else: return 'Very High Risk (75+)'

df['age_risk_group'] = df['age'].apply(age_risk)

# KPI 3: Clinical Complexity Score
# Combining number of diagnoses, medications, and procedures into a single score
df['clinical_complexity'] = (
    df['number_diagnoses'] / df['number_diagnoses'].max() * 0.4 +
    df['num_medications'] / df['num_medications'].max() * 0.3 +
    df['num_procedures'] / df['num_procedures'].max() * 0.3
) * 100

df['complexity_level'] = pd.cut(df['clinical_complexity'],
    bins=[0, 25, 50, 75, 100],
    labels=['Low', 'Medium', 'High', 'Very High'],
    include_lowest=True)

# KPI 4: Prior Visit Burden
df['prior_visits_total'] = df['number_inpatient'] + df['number_emergency'] + df['number_outpatient']
df['is_repeat_visitor'] = (df['prior_visits_total'] >= 2).astype(int)
df['repeat_visitor_label'] = df['is_repeat_visitor'].map({0: 'First/Low Contact', 1: 'Repeat Visitor'})

# KPI 5: Insulin Status (readable labels for Tableau)
insulin_map = {0: 'No Insulin', 1: 'Steady', 2: 'Decreased', 3: 'Increased'}
df['insulin_status'] = df['insulin'].map(insulin_map)

# KPI 6: A1C Test Status
a1c_map = {0: 'Not Tested', 1: 'Normal', 2: '>7 (Elevated)', 3: '>8 (High)'}
df['a1c_status'] = df['a1cresult'].map(a1c_map)

# KPI 7: Medication Change (readable)
df['med_changed'] = df['change'].map({'Ch': 'Medication Changed', 'No': 'No Change'})

# KPI 8: Length of Stay Category
def los_category(days):
    if days <= 2: return 'Short (1-2 days)'
    elif days <= 5: return 'Medium (3-5 days)'
    elif days <= 9: return 'Long (6-9 days)'
    else: return 'Extended (10+ days)'

df['stay_category'] = df['time_in_hospital'].apply(los_category)

# KPI 9: High-Risk Flag (composite)
df['high_risk_flag'] = (
    (df['age'] >= 70) &
    (df['number_diagnoses'] >= 7) &
    (df['number_inpatient'] >= 1)
).astype(int)
df['risk_flag_label'] = df['high_risk_flag'].map({0: 'Standard', 1: '⚠ High Risk'})

print("✅ All KPI columns computed:")
new_cols = ['readmission_status', 'age_risk_group', 'clinical_complexity', 'complexity_level',
            'prior_visits_total', 'is_repeat_visitor', 'repeat_visitor_label', 'insulin_status',
            'a1c_status', 'med_changed', 'stay_category', 'high_risk_flag', 'risk_flag_label']
for c in new_cols:
    print(f"  ✅ {c} ({df[c].dtype}) — {df[c].nunique()} unique values")

✅ All KPI columns computed:
  ✅ readmission_status (str) — 2 unique values
  ✅ age_risk_group (str) — 4 unique values
  ✅ clinical_complexity (float64) — 981 unique values
  ✅ complexity_level (category) — 4 unique values
  ✅ prior_visits_total (int64) — 34 unique values
  ✅ is_repeat_visitor (int64) — 2 unique values
  ✅ repeat_visitor_label (str) — 2 unique values
  ✅ insulin_status (str) — 4 unique values
  ✅ a1c_status (str) — 4 unique values
  ✅ med_changed (str) — 2 unique values
  ✅ stay_category (str) — 4 unique values
  ✅ high_risk_flag (int64) — 2 unique values
  ✅ risk_flag_label (str) — 2 unique values


## 3. KPI Summary Table
Validating computed KPIs against the baseline readmission rate.

In [4]:
baseline = df['readmitted_30day'].mean() * 100

print(f"Baseline 30-Day Readmission Rate: {baseline:.2f}%")
print("="*65)

# KPI breakdowns
kpi_groups = {
    'Age Risk Group': 'age_risk_group',
    'Complexity Level': 'complexity_level',
    'Repeat Visitor': 'repeat_visitor_label',
    'Insulin Status': 'insulin_status',
    'A1C Test Status': 'a1c_status',
    'Stay Category': 'stay_category',
    'High Risk Flag': 'risk_flag_label',
    'Medication Changed': 'med_changed',
}

for title, col in kpi_groups.items():
    print(f"\n--- {title} ---")
    summary = df.groupby(col).agg(
        patients=('readmitted_30day', 'count'),
        readmission_rate=('readmitted_30day', 'mean')
    )
    summary['readmission_rate'] = (summary['readmission_rate'] * 100).round(2)
    summary['lift_vs_baseline'] = (summary['readmission_rate'] / baseline).round(2)
    print(summary.to_string())

Baseline 30-Day Readmission Rate: 8.98%

--- Age Risk Group ---
                      patients  readmission_rate  lift_vs_baseline
age_risk_group                                                    
High Risk (60-74)        15689              9.02              1.00
Low Risk (<40)            4500              6.67              0.74
Medium Risk (40-59)      19179              7.23              0.81
Very High Risk (75+)     30622             10.40              1.16

--- Complexity Level ---
                  patients  readmission_rate  lift_vs_baseline
complexity_level                                              
Low                  21887              7.66              0.85
Medium               42267              9.54              1.06
High                  5793              9.77              1.09
Very High               43             23.26              2.59

--- Repeat Visitor ---
                      patients  readmission_rate  lift_vs_baseline
repeat_visitor_label                   

## 4. Drop Unnecessary Columns for Tableau
Remove encounter_id and patient_nbr (identifiers not useful for dashboard analysis).

In [5]:
drop_cols = ['encounter_id', 'patient_nbr']
df_tableau = df.drop(columns=drop_cols, errors='ignore')

print(f"Dropped: {drop_cols}")
print(f"Final Tableau dataset: {df_tableau.shape[0]:,} rows, {df_tableau.shape[1]} columns")
print(f"\nAll columns in Tableau dataset:")
for i, c in enumerate(df_tableau.columns, 1):
    print(f"  {i:02d}. {c}")

Dropped: ['encounter_id', 'patient_nbr']
Final Tableau dataset: 69,990 rows, 59 columns

All columns in Tableau dataset:
  01. race
  02. gender
  03. age
  04. admission_type_id
  05. discharge_disposition_id
  06. admission_source_id
  07. time_in_hospital
  08. payer_code
  09. medical_specialty
  10. num_lab_procedures
  11. num_procedures
  12. num_medications
  13. number_outpatient
  14. number_emergency
  15. number_inpatient
  16. diag_1
  17. diag_2
  18. diag_3
  19. number_diagnoses
  20. max_glu_serum
  21. a1cresult
  22. metformin
  23. repaglinide
  24. nateglinide
  25. chlorpropamide
  26. glimepiride
  27. acetohexamide
  28. glipizide
  29. glyburide
  30. tolbutamide
  31. pioglitazone
  32. rosiglitazone
  33. acarbose
  34. miglitol
  35. troglitazone
  36. tolazamide
  37. insulin
  38. glyburide_metformin
  39. glipizide_metformin
  40. glimepiride_pioglitazone
  41. metformin_rosiglitazone
  42. metformin_pioglitazone
  43. change
  44. diabetesmed
  45. age_b

## 5. Export Tableau-Ready Dataset

In [6]:
TABLEAU_PATH.parent.mkdir(parents=True, exist_ok=True)
df_tableau.to_csv(TABLEAU_PATH, index=False)

file_size = TABLEAU_PATH.stat().st_size / (1024 * 1024)
print(f"✅ Saved Tableau-ready dataset to: {TABLEAU_PATH}")
print(f"   Rows: {df_tableau.shape[0]:,}")
print(f"   Columns: {df_tableau.shape[1]}")
print(f"   File size: {file_size:.1f} MB")

✅ Saved Tableau-ready dataset to: /Users/omkarshukla/SectionB_G-15_diabetes/data/processed/tableau_ready.csv
   Rows: 69,990
   Columns: 59
   File size: 22.9 MB


## 6. Final Validation Check

In [7]:
print("=== FINAL VALIDATION ===")
print(f"Rows: {df_tableau.shape[0]:,}")
print(f"Columns: {df_tableau.shape[1]}")
print(f"Nulls: {df_tableau.isnull().sum().sum()}")
print(f"Target (readmitted_30day) distribution:")
print(df_tableau['readmitted_30day'].value_counts())
print(f"\n30-day Readmission Rate: {df_tableau['readmitted_30day'].mean()*100:.2f}%")
print(f"\nKPI columns present:")
kpi_cols = ['readmission_status', 'age_risk_group', 'clinical_complexity', 'complexity_level',
            'prior_visits_total', 'repeat_visitor_label', 'insulin_status', 'a1c_status',
            'med_changed', 'stay_category', 'risk_flag_label']
for c in kpi_cols:
    print(f"  {'✅' if c in df_tableau.columns else '❌'} {c}")

print(f"\n✅ Dataset is Tableau-ready. Load '{TABLEAU_PATH.name}' into Tableau Public.")

=== FINAL VALIDATION ===
Rows: 69,990
Columns: 59
Nulls: 0
Target (readmitted_30day) distribution:
readmitted_30day
0    63705
1     6285
Name: count, dtype: int64

30-day Readmission Rate: 8.98%

KPI columns present:
  ✅ readmission_status
  ✅ age_risk_group
  ✅ clinical_complexity
  ✅ complexity_level
  ✅ prior_visits_total
  ✅ repeat_visitor_label
  ✅ insulin_status
  ✅ a1c_status
  ✅ med_changed
  ✅ stay_category
  ✅ risk_flag_label

✅ Dataset is Tableau-ready. Load 'tableau_ready.csv' into Tableau Public.
